# Truth-conflict axis vs. deception-probe transfer — Colab runner

Runs the full pipeline from `docs/PLAN.md` on a **free Colab T4** GPU (16GB, 4-bit Qwen2.5-7B-Instruct).

**Before running:** Runtime > Change runtime type > GPU (T4). Free-tier sessions can idle-disconnect after ~90 min and hard-cap around 12h; every stage caches to Drive, so if you get disconnected, just re-run from the top — completed work is skipped, not redone.

**One-time setup:** upload `mats12_sim_dissim.zip` (the project source, no venv/cache/results/data) to the root of your Google Drive before running the cells below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, pathlib

PROJECT = pathlib.Path('/content/mats12_sim_dissim')
ZIP_IN_DRIVE = pathlib.Path('/content/drive/MyDrive/mats12_sim_dissim.zip')
DRIVE_STATE = pathlib.Path('/content/drive/MyDrive/mats12_sim_dissim_state')  # persists cache/results/data across sessions

if not PROJECT.exists():
    assert ZIP_IN_DRIVE.exists(), (
        f"Upload mats12_sim_dissim.zip to the root of My Drive first, then re-run this cell. "
        f"Looked for: {ZIP_IN_DRIVE}"
    )
    with zipfile.ZipFile(ZIP_IN_DRIVE) as z:
        z.extractall(PROJECT)
    print(f"extracted project to {PROJECT}")
else:
    print(f"project already present at {PROJECT}")

DRIVE_STATE.mkdir(parents=True, exist_ok=True)
for name in ("data", "cache", "results", "probes"):
    real = DRIVE_STATE / name
    real.mkdir(exist_ok=True)
    link = PROJECT / name
    if link.is_symlink() or link.exists():
        if not link.is_symlink():
            import shutil; shutil.rmtree(link) if link.is_dir() else link.unlink()
        else:
            link.unlink()
    os.symlink(real, link)
print("data/cache/results/probes now point into Drive — survives a disconnect.")

In [ ]:
%cd /content/mats12_sim_dissim
!pip install -q bitsandbytes>=0.43 scikit-learn scipy pandas matplotlib seaborn statsmodels tqdm

## 1. CPU smoke test (no model download, ~10s)
Confirms the pipeline itself runs correctly in this environment before spending GPU time.

In [ ]:
!python scripts/run_all.py --dry-run --synthetic

## 2. Tiny real-model dry run (validates the GPU/model path, a few minutes)

In [ ]:
!python scripts/run_all.py --dry-run --model qwen2.5-7b-instruct

## 3. Full primary run
~1,336 prompts, 4-bit Qwen2.5-7B on a T4. Activation caching is per-row and resumable — if the runtime disconnects, re-run this cell and it picks up where it left off.

In [ ]:
!python scripts/run_all.py --model qwen2.5-7b-instruct

In [ ]:
from IPython.display import Image, display
import json

rdir = PROJECT / 'results' / 'qwen2.5-7b-instruct'
for fig in ('validity.png', 'transfer_heatmap.png', 'monotonicity.png', 'direction_cosines.png'):
    p = rdir / fig
    if p.exists():
        print(fig); display(Image(str(p)))

print(json.dumps(json.loads((rdir / 'baselines.json').read_text()), indent=2))

## 4. (Optional) Replication on a second model
Llama-3.1-8B is gated — run `huggingface-cli login` with a token first (needs a HF account that has accepted the Llama license). Gemma-3-12b-it is open but larger; `--headline-only` drops the formal-style cells to cut compute.

In [ ]:
# from huggingface_hub import login; login()  # paste your HF token when prompted
# !python scripts/run_all.py --model llama-3.1-8b-instruct --headline-only